In [52]:
import pandas as pd 
import os
from scripts.parser_cotahist import ler_cotahist

path = ("dados_brutos/COTAHIST_A2024.TXT")

df = ler_cotahist(path, nrows=100000)
df.head()

,tipo_registro,data,codigo_acao,nome_empresa,preco_abertura,preco_fechamento,volume
0,00,COTAHIST,024BOVESPA 2,1231,NaN,NaN,NaN
1,01,20240102,AALR3,ALLIAR,1020.0,850.0,4.014875e+08
2,01,20240102,ABCB4,ABC BRASIL,2398.0,2288.0,4.494731e+09
3,01,20240102,ABEV3,AMBEV S/A,1372.0,1371.0,1.598391e+10
4,01,20240102,BBDC3,BRADESCO,1526.0,1511.0,6.857685e+09


In [53]:
#salvar o df em formato parquet
os.makedirs("bronze", exist_ok=True)

df.to_parquet("bronze/cotahist_2024.parquet", index=False)
print("O arquivo Parquet foi criado em: bronze/cotahist_2024.parquet")

O arquivo Parquet foi criado em: bronze/cotahist_2024.parquet


In [54]:
#dividir por 100 conforme manual da B3 para tratar decimais
df["preco_abertura"] = df["preco_abertura"] / 100
df["preco_fechamento"] = df["preco_fechamento"] / 100

#filtrar somento por acoes 
df = df[df["tipo_registro"] == "01"]

#converter para formato de datetime (sem o filtro de acoes estava pegando o cabeçalho e estava retornando erro)
df["data"] = pd.to_datetime(df["data"], format="%Y%m%d")

#Limpar espaços vazios nos textos
df["codigo_acao"] = df["codigo_acao"].str.strip()
df["nome_empresa"] = df["nome_empresa"].str.strip()


In [55]:
#salvar na camada silver
os.makedirs("silver", exist_ok=True)
df.to_parquet("silver/cotahist_2024_silver.parquet", index=False)
print("Dados limpos e tipados, camada silver concluida!")

df.head()

Dados limpos e tipados, camada silver concluida!


,tipo_registro,data,codigo_acao,nome_empresa,preco_abertura,preco_fechamento,volume
1,01,2024-01-02,AALR3,ALLIAR,10.20,8.50,4.014875e+08
2,01,2024-01-02,ABCB4,ABC BRASIL,23.98,22.88,4.494731e+09
3,01,2024-01-02,ABEV3,AMBEV S/A,13.72,13.71,1.598391e+10
4,01,2024-01-02,BBDC3,BRADESCO,15.26,15.11,6.857685e+09
5,01,2024-01-02,ALPA3,ALPARGATAS,10.11,10.00,2.912000e+06


In [64]:
df = df.sort_values(['nome_empresa'])
df['variacao_diaria'] = df.groupby('codigo_acao')[
    'preco_fechamento'].pct_change() * 100


df.head()

,tipo_registro,data,codigo_acao,nome_empresa,preco_abertura,preco_fechamento,volume,variacao_diaria
904,01,2024-01-02,MMMC34,3M,132.48,135.07,12162290.0,NaN
2580,01,2024-01-03,MMMC34,3M,135.07,133.00,1354980.0,-1.532539
4587,01,2024-01-04,MMMC34,3M,132.95,133.00,7850410.0,0.000000
6224,01,2024-01-05,MMMC34,3M,135.66,132.00,99256980.0,-0.751880
7896,01,2024-01-08,MMMC34,3M,132.55,132.70,7693410.0,0.530303
